> **Status of this notebook.** Revised for methodological soundness (stratified tuning subset, seeds, early stopping, fresh tuner state on every run, a trivial-baseline reference, honest notes on search noise), but TensorFlow and Keras Tuner were not available when the revisions were made, so **the notebook has not been re-run here and all outputs are cleared**. Install `requirements-deep-learning.txt` and run it top to bottom (a GPU runtime such as Colab is recommended) to regenerate every result.

# Hyperparameter Tuning in ANN (Keras + Keras Tuner)


In real-world applications, hyperparameters are tuned on a **subset of the training data** to save computation, and the best configuration is then retrained on the full training set.

### Workflow
1. Draw a **stratified** 10k-image tuning subset from the training data (class balance preserved).
2. Run Keras Tuner's `RandomSearch` on it, scoring each trial on a validation split carved out of that subset, with early stopping.
3. Retrain the best configuration on the full training set, using a validation split from the **training** data to pick the epoch (again with early stopping).
4. Evaluate **once** on the untouched test set.

The test set is never used by the tuner or by early stopping.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

# Load Fashion MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_test = (x_train / 255.0).astype("float32"), (x_test / 255.0).astype("float32")

# Flatten for ANN
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Stratified 10k subset for tuning (class balance preserved)
x_tune, _, y_tune, _ = train_test_split(x_train, y_train, train_size=10000, stratify=y_train, random_state=SEED)
print("tuning subset:", x_tune.shape, "| class counts:", np.bincount(y_tune))
print("Reference: a random guess scores 10 % on this balanced 10-class problem.")

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(784,)))

    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(
            layers.Dense(
                units=hp.Int(f"units_{i}", min_value=32, max_value=256, step=32),
                activation="relu"
            )
        )

    model.add(layers.Dense(10, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("lr", [1e-2, 1e-3, 1e-4])
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    seed=SEED,
    directory="realistic_ann_tuner",
    project_name="fashion_ann_subset",
    overwrite=True,          # always start from a clean search; stale trials would otherwise be silently reused
)

In [ ]:
tuner.search(x_tune, y_tune, epochs=20, validation_split=0.2, callbacks=[early_stop])

## Best Hyperparameters Found

Each trial is scored on a 2,000-image validation split from one training run, so differences of a point or so between trials are within noise, and 5 random trials explore only a small part of the space. Treat the winner as "a good configuration", not "the optimum".

In [ ]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print("Best Hyperparameters:")
print(f"Number of Layers: {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"Units in Layer {i+1}: {best_hps.get(f'units_{i}')}")
print(f"Learning Rate: {best_hps.get('lr')}")

## Final Model Training on Full Dataset
Now the best configuration is built from scratch and trained on the **entire training set (60,000 samples)**; a 10 % validation split of the training data drives early stopping.

In [ ]:
final_model = tuner.hypermodel.build(best_hps)
final_model.fit(x_train, y_train, epochs=30, validation_split=0.1, callbacks=[early_stop])

In [ ]:
# Single evaluation on the untouched test set
test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

### Baseline reference

A tuned network should be compared with something simple. A multinomial logistic regression on the same pixels is the natural linear baseline; it is trained on the same 10k tuning subset and scored on the same test set.

In [ ]:
from sklearn.linear_model import LogisticRegression
baseline = LogisticRegression(max_iter=300).fit(x_tune, y_tune)
print(f"Logistic-regression baseline (10k train rows) test accuracy: {baseline.score(x_test, y_test):.4f}")

## Limitations (Fashion-MNIST ANN)

- One search of 5 random trials and one final training run: no repeated seeds, so run-to-run variation is not measured.
- The search is on a 10k subset; the best configuration there is not guaranteed to be best at 60k.
- Fashion-MNIST is a clean benchmark; the numbers do not transfer to real-world images.

# CNN Hyperparameter Tuning on CIFAR-10 with Keras Tuner


The same workflow for a CNN on CIFAR-10: stratified 10k tuning subset, `RandomSearch`, retraining on the full training set with early stopping, one test evaluation.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train, x_test = (x_train / 255.0).astype("float32"), (x_test / 255.0).astype("float32")
y_train, y_test = y_train.ravel(), y_test.ravel()

# Stratified 10k subset for tuning
x_tune, _, y_tune, _ = train_test_split(x_train, y_train, train_size=10000, stratify=y_train, random_state=SEED)
print("tuning subset:", x_tune.shape, "| class counts:", np.bincount(y_tune))

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

In [ ]:
def build_model(hp):
    inputs = keras.Input(shape=(32, 32, 3))
    x = inputs

    # Convolutional blocks
    for i in range(hp.Int("num_conv_blocks", 1, 3)):
        x = layers.Conv2D(
            filters=hp.Int(f"filters_{i}", 32, 128, step=32),
            kernel_size=3,
            padding="same",
            activation="relu"
        )(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)

    # Dense layer
    x = layers.Dense(
        units=hp.Int("dense_units", 64, 256, step=64),
        activation="relu"
    )(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("lr", [1e-2, 1e-3, 1e-4])
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    seed=SEED,
    directory="cnn_tuner_results",
    project_name="cifar10_cnn2",
    overwrite=True,
)

In [ ]:
tuner.search(x_tune, y_tune, epochs=20, validation_split=0.2, callbacks=[early_stop])

## Best Hyperparameters from Tuner

In [ ]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print("Best Hyperparameters:")
print(f"Conv Blocks: {best_hps.get('num_conv_blocks')}")
for i in range(best_hps.get('num_conv_blocks')):
    print(f"Filters in Block {i+1}: {best_hps.get(f'filters_{i}')}")
print(f"Dense Units: {best_hps.get('dense_units')}")
print(f"Learning Rate: {best_hps.get('lr')}")

## Final Model Training on Full CIFAR-10 Training Set
The best configuration is rebuilt and trained from scratch on the **entire training set (50,000 samples)**, with a 10 % validation split of the training data for early stopping.

In [ ]:
final_model = tuner.hypermodel.build(best_hps)
final_model.fit(x_train, y_train, epochs=30, validation_split=0.1, callbacks=[early_stop])

In [ ]:
# Single evaluation on the untouched test set
test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}   (random guess = 0.10)")

## Limitations (CIFAR-10 CNN)

- Same caveats as above: one search, one run, validation scores from a 2,000-image split.
- The search space is small (1-3 conv blocks, no augmentation, no regularisation). For stronger results see the `transfer-learning` project.
- Tuner output directories (`realistic_ann_tuner/`, `cnn_tuner_results/`) are generated artefacts and are git-ignored.